### 🧠 Reading the Data and First Intuition

At this stage, the goal is to build an initial understanding of the dataset before performing any analysis or modeling. By reading the data, we explore its structure — the number of rows and columns, the meaning of each feature, and the types of data (numerical, categorical, text, or dates). This helps us identify whether the dataset is clean, contains missing values, or includes unusual entries.

Our first intuition comes from visually inspecting a few records and basic statistics such as means, ranges, and unique values. This early look often reveals important clues: which features might influence the target variable, whether scaling or encoding will be needed, and if potential outliers or data quality issues exist. Essentially, this step forms a mental map of the data, guiding the direction of preprocessing and analysis that follows.

In [51]:
import pandas as pd
df=pd.read_csv('/content/train_u6lujuX_CVtuZ9i (1).csv')
df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [52]:
df.isna().sum()

,0
Loan_ID,0
Gender,13
Married,3
Dependents,15
Education,0
Self_Employed,32
ApplicantIncome,0
CoapplicantIncome,0
LoanAmount,22
Loan_Amount_Term,14


In [53]:
threshold=0.05*len(df)

In [54]:
threshold
df.drop('Loan_ID',axis=1,inplace=True)

In [55]:
df.dtypes

,0
Gender,object
Married,object
Dependents,object
Education,object
Self_Employed,object
ApplicantIncome,int64
CoapplicantIncome,float64
LoanAmount,float64
Loan_Amount_Term,float64
Credit_History,float64


In [56]:

print(df.isna().sum())

for col in df.select_dtypes('object'):
    mode = df[col].mode()[0]
    df[col].fillna(mode, inplace=True)

for col in df.select_dtypes(['int64','float64']):
    median = df[col].median()
    df[col].fillna(median, inplace=True)
print(df.isna().sum())


Gender               13
Married               3
Dependents           15
Education             0
Self_Employed        32
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount           22
Loan_Amount_Term     14
Credit_History       50
Property_Area         0
Loan_Status           0
dtype: int64
Gender               0
Married              0
Dependents           0
Education            0
Self_Employed        0
ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Loan_Amount_Term     0
Credit_History       0
Property_Area        0
Loan_Status          0
dtype: int64


/tmp/ipython-input-2651030116.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(mode, inplace=True)
/tmp/ipython-input-2651030116.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method

### 🔠 Label Encoding

I applied label encoding to convert the categorical features into numeric form using `LabelEncoder` from `sklearn.preprocessing`. This step ensured that all the categorical variables were transformed into numerical values, allowing the model to process them efficiently.

In [57]:
from sklearn.preprocessing import LabelEncoder

binary_cols = ['Gender', 'Married', 'Education', 'Self_Employed','Loan_Status']
le = LabelEncoder()
for col in binary_cols:
    df[col] = le.fit_transform(df[col])

df = pd.get_dummies(df, columns=['Property_Area', 'Dependents'], drop_first=True)
print(df.head())


   Gender  Married  Education  Self_Employed  ApplicantIncome  \
0       1        0          0              0             5849   
1       1        1          0              0             4583   
2       1        1          0              1             3000   
3       1        1          1              0             2583   
4       1        0          0              0             6000   

   CoapplicantIncome  LoanAmount  Loan_Amount_Term  Credit_History  \
0                0.0       128.0             360.0             1.0   
1             1508.0       128.0             360.0             1.0   
2                0.0        66.0             360.0             1.0   
3             2358.0       120.0             360.0             1.0   
4                0.0       141.0             360.0             1.0   

   Loan_Status  Property_Area_Semiurban  Property_Area_Urban  Dependents_1  \
0            1                    False                 True         False   
1            0                  

In [58]:
df.shape

(614, 15)

### Feature Scaling

I applied feature scaling using `StandardScaler` from `sklearn.preprocessing` to standardize the numerical features. This step transformed the data so that each feature has a mean of 0 and a standard deviation of 1, ensuring all variables are on a similar scale for better model performance.

In [59]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

num_cols = ['ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History']

df[num_cols] = scaler.fit_transform(df[num_cols])


In [60]:
from sklearn.model_selection import train_test_split

X = df.drop('Loan_Status', axis=1)
y = df['Loan_Status']


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train distribution:\n", y_train.value_counts())
print("y_test distribution:\n", y_test.value_counts())


X_train shape: (429, 14)
X_test shape: (185, 14)
y_train distribution:
 Loan_Status
1    295
0    134
Name: count, dtype: int64
y_test distribution:
 Loan_Status
1    127
0     58
Name: count, dtype: int64


### ⚙️ Logistic Regression (L1 and L2 Regularization)

I trained logistic regression models using both **L1** and **L2** regularization while keeping the other hyperparameters constant. This was done to compare how each regularization type affects model performance and feature selection.

In [61]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
penalties = ['l1','l2']

for p in penalties:
    print(f"\n--- Logistic Regression with {p.upper()} penalty ---")
    model = LogisticRegression(penalty=p, solver='liblinear', random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Classification Report:\n", classification_report(y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))



--- Logistic Regression with L1 penalty ---
Accuracy: 0.8540540540540541
Classification Report:
               precision    recall  f1-score   support

           0       0.94      0.57      0.71        58
           1       0.83      0.98      0.90       127

    accuracy                           0.85       185
   macro avg       0.89      0.78      0.81       185
weighted avg       0.87      0.85      0.84       185

Confusion Matrix:
 [[ 33  25]
 [  2 125]]

--- Logistic Regression with L2 penalty ---
Accuracy: 0.8540540540540541
Classification Report:
               precision    recall  f1-score   support

           0       0.92      0.59      0.72        58
           1       0.84      0.98      0.90       127

    accuracy                           0.85       185
   macro avg       0.88      0.78      0.81       185
weighted avg       0.86      0.85      0.84       185

Confusion Matrix:
 [[ 34  24]
 [  3 124]]


### 🔍 Logistic Regression with Grid Search

I used **GridSearchCV** to tune the hyperparameters of the Logistic Regression model and identify the best combination for optimal accuracy. The search included different solvers (`lbfgs`, `liblinear`, and `saga`) with various penalties (`l1`, `l2`, and `elasticnet`) and regularization strengths (`C = 0.1, 1.0`). A 5-fold cross-validation was performed to evaluate each configuration. After fitting the model, the best estimator and its corresponding accuracy on the test set were reported.

In [63]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_breast_cancer

param_grid = [
    {'solver': ['lbfgs'], 'penalty': ['l2'], 'C': [0.1, 1.0], 'max_iter': [500]},
    {'solver': ['liblinear'], 'penalty': ['l1', 'l2'], 'C': [0.1, 1.0], 'max_iter': [500]},
    {'solver': ['saga'], 'penalty': ['elasticnet'], 'l1_ratio': [0.5], 'C': [0.1, 1.0], 'max_iter': [500]},
]

log_reg = LogisticRegression()

grid = GridSearchCV(
    estimator=log_reg,
    param_grid=param_grid,
    scoring='accuracy',
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)


print(f"\nBest CV accuracy: {grid.best_score_:.4f}")
print(f"Best hyperparameters: {grid.best_params_}")

best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {test_acc:.4f}")


Fitting 5 folds for each of 8 candidates, totalling 40 fits

Best CV accuracy: 0.7924
Best hyperparameters: {'C': 0.1, 'max_iter': 500, 'penalty': 'l2', 'solver': 'lbfgs'}
Test accuracy: 0.8541


### 🧪 Logistic Regression Manual Trials

I manually tested several Logistic Regression configurations by varying the penalty type, regularization strength (`C`), solver, and maximum iterations. Each trial was trained on the same data and evaluated using accuracy on the test set. The results from all runs were compared, and the model with the highest test accuracy was selected as the best-performing configuration.

In [62]:

trials = [
    {"penalty": "l2", "C": 1.0, "solver": "lbfgs", "max_iter": 100},
    {"penalty": "l1", "C": 0.5, "solver": "liblinear", "max_iter": 200},
    {"penalty": "l2", "C": 0.1, "solver": "saga", "max_iter": 500},
    {"penalty": "elasticnet", "C": 1.0, "l1_ratio": 0.5, "solver": "saga", "max_iter": 1000},
]

results = []
for i, params in enumerate(trials, 1):
    model = LogisticRegression(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results.append((i, params, acc))
    print(f"Trial {i}: Accuracy = {acc:.4f}, Params = {params}")

best = max(results, key=lambda x: x[2])
print(f"\nBest trial: {best[0]} with accuracy {best[2]:.4f}")
print(f"Best hyperparameters: {best[1]}")


Trial 1: Accuracy = 0.8486, Params = {'penalty': 'l2', 'C': 1.0, 'solver': 'lbfgs', 'max_iter': 100}
Trial 2: Accuracy = 0.8541, Params = {'penalty': 'l1', 'C': 0.5, 'solver': 'liblinear', 'max_iter': 200}
Trial 3: Accuracy = 0.8541, Params = {'penalty': 'l2', 'C': 0.1, 'solver': 'saga', 'max_iter': 500}
Trial 4: Accuracy = 0.8541, Params = {'penalty': 'elasticnet', 'C': 1.0, 'l1_ratio': 0.5, 'solver': 'saga', 'max_iter': 1000}

Best trial: 2 with accuracy 0.8541
Best hyperparameters: {'penalty': 'l1', 'C': 0.5, 'solver': 'liblinear', 'max_iter': 200}


### 📊 Evaluation Function

I defined a custom function `print_score()` to evaluate the performance of classification models on both the training and test sets. It displays key metrics such as **accuracy**, a detailed **classification report** (precision, recall, and F1-score), and the **confusion matrix**. This function helps in quickly assessing how well a model performs and whether it may be overfitting or underfitting.

In [65]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

def print_score(clf, X_train, y_train, X_test, y_test, train=True):
    if train:
        pred = clf.predict(X_train)
        clf_report = pd.DataFrame(classification_report(y_train, pred, output_dict=True))
        print("Train Result:\n================================================")
        print(f"Accuracy Score: {accuracy_score(y_train, pred) * 100:.2f}%")
        print("_______________________________________________")
        print(f"CLASSIFICATION REPORT:\n{clf_report}")
        print("_______________________________________________")
        print(f"Confusion Matrix: \n {confusion_matrix(y_train, pred)}\n")

    elif train==False:
        pred = clf.predict(X_test)
        clf_report = pd.DataFrame(classification_report(y_test, pred, output_dict=True))
        print("Test Result:\n================================================")
        print(f"Accuracy Score: {accuracy_score(y_test, pred) * 100:.2f}%")
        print("_______________________________________________")
        print(f"CLASSIFICATION REPORT:\n{clf_report}")
        print("_______________________________________________")
        print(f"Confusion Matrix: \n {confusion_matrix(y_test, pred)}\n")

### 🌳 Decision Tree Manual Hyperparameter Trials

I manually tested several Decision Tree configurations by varying key hyperparameters such as **criterion**, **splitter**, **max depth**, **minimum samples split**, and **minimum samples per leaf**. For each trial, the model was trained and evaluated using the custom `print_score()` function to observe both training and testing performance.

The results from all trials were stored in a DataFrame and sorted by test accuracy to identify the best-performing model. The optimal parameter combination was then used to retrain the final Decision Tree classifier for further analysis.

In [67]:

from sklearn.tree import DecisionTreeClassifier

param_list = [
    {"criterion": "gini", "splitter": "best", "max_depth": 3, "min_samples_split": 2, "min_samples_leaf": 1},
    {"criterion": "entropy", "splitter": "random", "max_depth": 5, "min_samples_split": 3, "min_samples_leaf": 2},
    {"criterion": "gini", "splitter": "best", "max_depth": 8, "min_samples_split": 2, "min_samples_leaf": 4},
    {"criterion": "entropy", "splitter": "best", "max_depth": 10, "min_samples_split": 4, "min_samples_leaf": 1},
    {"criterion": "gini", "splitter": "random", "max_depth": 15, "min_samples_split": 2, "min_samples_leaf": 3}
]
results = []
for i, param in enumerate (param_list,1):
  print(f"\n========== Trial {i} ==========")
  tree_clf = DecisionTreeClassifier(**param)
  tree_clf.fit(X_train, y_train)
  print_score(tree_clf, X_train, y_train, X_test, y_test, train=True)
  print_score(tree_clf, X_train, y_train, X_test, y_test, train=False)
  train_acc = accuracy_score(y_train, tree_clf.predict(X_train))
  test_acc = accuracy_score(y_test, tree_clf.predict(X_test))
  results.append({
        "Trial": i,
        "Params": param,
        "Train Accuracy": train_acc,
        "Test Accuracy": test_acc
  })
results_df = pd.DataFrame(results)


results_df = results_df.sort_values(by="Test Accuracy", ascending=False).reset_index(drop=True)
best_params = results_df.iloc[0]["Params"]
print("\n Best Parameters:", best_params)
print("\n trial number : ",results_df.iloc[0]["Trial"])

best_model = DecisionTreeClassifier(**best_params, random_state=42)
best_model.fit(X_train, y_train)

print("\n Best model trained and ready for use!")


========== Trial 1 ==========
Train Result:
Accuracy Score: 80.65%
_______________________________________________
CLASSIFICATION REPORT:
                    0           1  accuracy   macro avg  weighted avg
precision    0.918033    0.788043  0.806527    0.853038      0.828646
recall       0.417910    0.983051  0.806527    0.700481      0.806527
f1-score     0.574359    0.874811  0.806527    0.724585      0.780964
support    134.000000  295.000000  0.806527  429.000000    429.000000
_______________________________________________
Confusion Matrix: 
 [[ 56  78]
 [  5 290]]

Test Result:
Accuracy Score: 80.00%
_______________________________________________
CLASSIFICATION REPORT:
                   0           1  accuracy   macro avg  weighted avg
precision   0.744186    0.816901       0.8    0.780544      0.794104
recall      0.551724    0.913386       0.8    0.732555      0.800000
f1-score    0.633663    0.862454       0.8    0.748058      0.790725
support    58.000000  127.000000    

### 🌿 Decision Tree – Grid Search Optimization

To automate hyperparameter tuning for the Decision Tree model, I used **GridSearchCV** to explore multiple combinations of parameters, including:

* **criterion:** `"gini"`, `"entropy"`
* **splitter:** `"best"`, `"random"`
* **max_depth:** range from 1 to 19
* **min_samples_split:** 2, 3, 4
* **min_samples_leaf:** range from 1 to 19

The grid search evaluated each configuration using **3-fold cross-validation**, optimizing for the **accuracy** metric.
After identifying the best parameters, the final Decision Tree model was retrained using these optimal settings, and its performance was evaluated on both training and testing data using the `print_score()` function.

---


In [68]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

params = {
    "criterion":("gini", "entropy"),
    "splitter":("best", "random"),
    "max_depth":(list(range(1, 20))),
    "min_samples_split":[2, 3, 4],
    "min_samples_leaf":list(range(1, 20)),
}


tree_clf = DecisionTreeClassifier(random_state=42)
tree_cv = GridSearchCV(tree_clf, params, scoring="accuracy", n_jobs=-1, verbose=1, cv=3)
tree_cv.fit(X_train, y_train)
best_params = tree_cv.best_params_
print(f"Best paramters: {best_params})")

tree_clf = DecisionTreeClassifier(**best_params)
tree_clf.fit(X_train, y_train)
print_score(tree_clf, X_train, y_train, X_test, y_test, train=True)
print_score(tree_clf, X_train, y_train, X_test, y_test, train=False)

Fitting 3 folds for each of 4332 candidates, totalling 12996 fits
Best paramters: {'criterion': 'gini', 'max_depth': 5, 'min_samples_leaf': 8, 'min_samples_split': 2, 'splitter': 'random'})
Train Result:
Accuracy Score: 79.25%
_______________________________________________
CLASSIFICATION REPORT:
                    0           1  accuracy   macro avg  weighted avg
precision    0.792208    0.792614  0.792541    0.792411      0.792487
recall       0.455224    0.945763  0.792541    0.700493      0.792541
f1-score     0.578199    0.862442  0.792541    0.720321      0.773658
support    134.000000  295.000000  0.792541  429.000000    429.000000
_______________________________________________
Confusion Matrix: 
 [[ 61  73]
 [ 16 279]]

Test Result:
Accuracy Score: 82.70%
_______________________________________________
CLASSIFICATION REPORT:
                   0           1  accuracy   macro avg  weighted avg
precision   0.825000    0.827586  0.827027    0.826293      0.826775
recall      0.5